Cell 1 — Imports & Configuration

In [46]:
import os
import json
import ast
import pandas as pd
from groq import Groq
from datetime import datetime, date
from IPython.display import display, Markdown

# ── Install groq if needed ──────────────────────────────────────────────────
# Run this in a separate cell if not installed:
# !pip install groq

# ── Set your FREE Groq API key ──────────────────────────────────────────────
os.environ["GROQ_API_KEY"] = "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"  # ← Paste your Groq key here

MODEL = "llama-3.3-70b-versatile"  # Free model on Groq, supports tool calling

client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("✅ Imports complete")
print(f"🔑 Groq API Key set: {os.environ['GROQ_API_KEY'][:12]}...")
print(f"🤖 Model: {MODEL}")

✅ Imports complete
🔑 Groq API Key set: gsk_ELTyHeNU...
🤖 Model: llama-3.3-70b-versatile


Cell 2 — Load Data

In [47]:
# ── Load CSVs ───────────────────────────────────────────────────────────────
INVENTORY_PATH = "product_inventory.csv"
ORDERS_PATH    = "orders.csv"
POLICY_PATH    = "policy.txt"

inventory_df = pd.read_csv(INVENTORY_PATH)
orders_df    = pd.read_csv(ORDERS_PATH)

with open(POLICY_PATH, "r") as f:
    POLICY_TEXT = f.read()

# ── Parse stock_per_size from string dict to real dict ──────────────────────
def parse_stock(val):
    if isinstance(val, dict):
        return val
    try:
        return ast.literal_eval(val)
    except Exception:
        return {}

inventory_df["stock_per_size"] = inventory_df["stock_per_size"].apply(parse_stock)

print(f"✅ Loaded {len(inventory_df)} products, {len(orders_df)} orders")
print(f"\n📋 Inventory columns: {inventory_df.columns.tolist()}")
print(f"📋 Orders columns:    {orders_df.columns.tolist()}")
display(inventory_df.head(3))

✅ Loaded 100 products, 100 orders

📋 Inventory columns: ['product_id', 'title', 'vendor', 'price', 'compare_at_price', 'tags', 'sizes_available', 'stock_per_size', 'is_sale', 'is_clearance', 'bestseller_score']
📋 Orders columns:    ['order_id', 'order_date', 'product_id', 'size', 'price_paid', 'customer_id']


,product_id,title,vendor,price,compare_at_price,tags,sizes_available,stock_per_size,is_sale,is_clearance,bestseller_score
0,P0001,Silk Avenue Style 1,Silk Avenue,137,137,"cocktail,lace,flowy",8|4|14|2|10,"{'8': 18, '4': 13, '14': 1, '2': 0, '10': 2}",False,False,72
1,P0002,Velour House Style 2,Velour House,446,506,"evening,cocktail,modest",16|10|6|2,"{'16': 5, '10': 13, '6': 10, '2': 8}",True,True,49
2,P0003,Aurelia Couture Style 3,Aurelia Couture,263,303,"lace,bridal,prom",10|14|2|8|12|4|16,"{'10': 17, '14': 9, '2': 20, '8': 19, '12': 11...",True,False,99


Cell 3 — Tool Implementation (Data Layer)

These are the **real** functions that retrieve data. The model never sees raw CSV — it only receives structured JSON responses from these functions.

In [48]:
def search_products(filters: dict) -> dict:
    """
    Search inventory with multi-constraint filtering.
    Supported filters:
      - tags          : str or list of str
      - max_price     : float
      - min_price     : float
      - size          : str or int
      - is_sale       : bool (or string "true"/"false")
      - is_clearance  : bool (or string "true"/"false")
      - vendor        : str
      - limit         : int (default 5)
    """
    df = inventory_df.copy()

    # Helper to safely parse bool from string or bool
    def to_bool(val):
        if isinstance(val, bool):
            return val
        if isinstance(val, str):
            return val.lower() in ("true", "1", "yes")
        return bool(val)

    # Tag filter
    if "tags" in filters and filters["tags"]:
        tag_input = filters["tags"]
        if isinstance(tag_input, str):
            tag_input = [tag_input]
        mask = df["tags"].apply(
            lambda t: any(tag.lower() in str(t).lower() for tag in tag_input)
        )
        df = df[mask]

    # Price filters
    if "max_price" in filters:
        df = df[df["price"] <= float(filters["max_price"])]
    if "min_price" in filters:
        df = df[df["price"] >= float(filters["min_price"])]

    # Size + stock filter
    if "size" in filters:
        size = str(filters["size"])
        def has_stock_in_size(row):
            stock = row["stock_per_size"]
            return stock.get(size, 0) > 0
        df = df[df.apply(has_stock_in_size, axis=1)]

    # Sale / clearance filters
    if "is_sale" in filters:
        df = df[df["is_sale"] == to_bool(filters["is_sale"])]
    if "is_clearance" in filters:
        df = df[df["is_clearance"] == to_bool(filters["is_clearance"])]

    # Vendor filter
    if "vendor" in filters and filters["vendor"]:
        df = df[df["vendor"].str.lower() == filters["vendor"].lower()]

    # Sort by bestseller_score
    df = df.sort_values("bestseller_score", ascending=False)

    limit = int(filters.get("limit", 5))
    results = []
    for _, row in df.head(limit).iterrows():
        size_key = str(filters.get("size", ""))
        stock_for_size = row["stock_per_size"].get(size_key, "N/A") if size_key else "varies"
        results.append({
            "product_id"      : row["product_id"],
            "title"           : row["title"],
            "vendor"          : row["vendor"],
            "price"           : row["price"],
            "compare_at_price": row["compare_at_price"],
            "tags"            : row["tags"],
            "is_sale"         : bool(row["is_sale"]),
            "is_clearance"    : bool(row["is_clearance"]),
            "bestseller_score": int(row["bestseller_score"]),
            "sizes_available" : row["sizes_available"],
            "stock_for_requested_size": stock_for_size,
        })

    return {
        "total_found": len(df),
        "returned"   : len(results),
        "filters_applied": filters,
        "results"    : results,
    }

print("✅ search_products fixed for Groq")

✅ search_products fixed for Groq


Cell 4 — Tool Schema (for Claude)

These JSON schemas tell Claude what tools exist and when to call them.

In [49]:
TOOLS = [
    {
        "name": "search_products",
        "description": (
            "Search the product inventory with one or more filters. "
            "Use this when the customer asks for product recommendations, browsing, or filtering. "
            "Always filter by size AND stock availability when the customer specifies a size."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "tags"        : {"type": "string", "description": "Style tag to filter by, e.g. 'modest', 'evening', 'cocktail', 'lace'"},
                "max_price"   : {"type": "number", "description": "Maximum price in USD as a number, e.g. 300"},
                "min_price"   : {"type": "number", "description": "Minimum price in USD as a number, e.g. 100"},
                "size"        : {"type": "string", "description": "Clothing size as string e.g. '8', '10', '14'"},
                "is_sale"     : {"type": "boolean", "description": "True to filter only sale items"},
                "is_clearance": {"type": "boolean", "description": "True to filter only clearance items"},
                "vendor"      : {"type": "string", "description": "Filter by vendor name"},
                "limit"       : {"type": "integer", "description": "Max number of results (default 5)"},
            },
            "required": [],
        },
    },
    {
        "name": "get_product",
        "description": (
            "Fetch complete product details for a specific product_id. "
            "Use this when you need detailed info about one specific product."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string", "description": "The product ID, e.g. P0001"},
            },
            "required": ["product_id"],
        },
    },
    {
        "name": "get_order",
        "description": (
            "Fetch an order by order_id. Returns order date, product, size, price paid. "
            "Use this before evaluating a return request."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID, e.g. O0043"},
            },
            "required": ["order_id"],
        },
    },
    {
        "name": "evaluate_return",
        "description": (
            "Evaluate whether an order is eligible for return or exchange based on return policy. "
            "Always use this tool when a customer asks about returning or exchanging an order."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID to evaluate, e.g. O0043"},
            },
            "required": ["order_id"],
        },
    },
]

print(f"✅ {len(TOOLS)} tools registered: {[t['name'] for t in TOOLS]}")

✅ 4 tools registered: ['search_products', 'get_product', 'get_order', 'evaluate_return']


Cell 5 — Tool Dispatcher

Routes Claude's tool_use requests to the correct Python function.

In [51]:
TOOL_REGISTRY = {
    "search_products": search_products,
    "get_product"    : get_product,
    "get_order"      : get_order,
    "evaluate_return": evaluate_return,
}

def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    """Call the correct tool function and return result as JSON string."""
    fn = TOOL_REGISTRY.get(tool_name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})
    
    # search_products expects a single 'filters' dict
    # Groq passes arguments flat, so we need to wrap them
    if tool_name == "search_products":
        result = fn(filters=tool_input)
    else:
        result = fn(**tool_input)
    
    return json.dumps(result, default=str)

print("✅ Dispatcher fixed for Groq")

✅ Dispatcher fixed for Groq


Cell 6 — Agent Core (Agentic Loop)

The agent loop:
1. Sends user message to Claude with tool definitions
2. If Claude requests a tool → dispatches it → sends result back
3. Repeats until Claude produces a final text response
4. Never hallucinates — all facts come from tool results

In [52]:
SYSTEM_PROMPT = f"""\
You are a Retail AI Assistant for a fashion boutique. You have two roles:

1. PERSONAL SHOPPER (Revenue Agent)
   - Help customers find products matching their style, size, budget, and preferences.
   - Always check actual stock before recommending (use search_products with size filter).
   - Prioritise sale items when requested. Sort by bestseller_score.
   - Explain your reasoning: why does this product fit the customer's constraints?
   - Never recommend a product without confirming it has stock in the requested size.

2. CUSTOMER SUPPORT (Operations Agent)
   - Handle return and exchange requests.
   - Always call evaluate_return(order_id) — never guess or apply policy from memory.
   - If an order or product does not exist, say so clearly. Do not invent data.
   - Explain the exact policy rule that applies and why.

ANTI-HALLUCINATION RULES:
- Never state product names, prices, or stock levels without calling a tool first.
- Never state return eligibility without calling evaluate_return.
- If a tool returns an error, relay that error to the customer politely.
- Do not fabricate order IDs, product IDs, or any data.

RETURN POLICY:
{POLICY_TEXT}

Today's date: {date.today().isoformat()}
"""


def run_agent(user_message: str, verbose: bool = True) -> str:
    """Run the agentic loop for a single user message using Groq."""

    messages = [{"role": "user", "content": user_message}]

    if verbose:
        print(f"👤 User: {user_message}")
        print("─" * 60)

    # Convert tools to Groq/OpenAI format
    groq_tools = []
    for tool in TOOLS:
        groq_tools.append({
            "type": "function",
            "function": {
                "name"       : tool["name"],
                "description": tool["description"],
                "parameters" : tool["input_schema"],
            }
        })

    max_iterations = 10
    iteration      = 0

    while iteration < max_iterations:
        iteration += 1

        try:
            response = client.chat.completions.create(
                model       = MODEL,
                messages    = [{"role": "system", "content": SYSTEM_PROMPT}] + messages,
                tools       = groq_tools,
                tool_choice = "auto",
                max_tokens  = 2048,
                temperature = 0,
            )
        except Exception as e:
            # ✅ FIXED: do NOT ask model to guess — surface the error directly
            error_msg = f"I'm sorry, I encountered a technical error and cannot process your request right now. Details: {str(e)}"
            if verbose:
                print(f"⚠️ API Error: {str(e)}")
                print(f"🤖 Assistant:\n{error_msg}")
            return error_msg

        msg           = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        # ── Tool calls ───────────────────────────────────────────────────────
        if finish_reason == "tool_calls" and msg.tool_calls:

            messages.append({
                "role"      : "assistant",
                "content"   : msg.content or "",
                "tool_calls": [
                    {
                        "id"      : tc.id,
                        "type"    : "function",
                        "function": {
                            "name"     : tc.function.name,
                            "arguments": tc.function.arguments,
                        }
                    }
                    for tc in msg.tool_calls
                ]
            })

            for tc in msg.tool_calls:
                tool_name = tc.function.name

                try:
                    tool_input = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    tool_input = {}

                if verbose:
                    print(f"🔧 Tool call: {tool_name}({json.dumps(tool_input)})")

                result_str = dispatch_tool(tool_name, tool_input)

                if verbose:
                    preview = result_str[:300] + "..." if len(result_str) > 300 else result_str
                    print(f"   ↳ Result: {preview}")

                messages.append({
                    "role"        : "tool",
                    "tool_call_id": tc.id,
                    "content"     : result_str,
                })

        # ── Final response ───────────────────────────────────────────────────
        else:
            final_text = msg.content or ""
            if verbose:
                print(f"🤖 Assistant:\n{final_text}")
            return final_text

    return "Max iterations reached. Please try again."


print("✅ Groq Agent core ready")

✅ Groq Agent core ready


Cell 7 — Interactive Chat Interface

Multi-turn conversation with the agent. Type `quit` or `exit` to stop.

In [53]:
def chat():
    """Run an interactive multi-turn chat session."""
    print("=" * 60)
    print("🛍️  Retail AI Assistant")
    print("   Personal Shopper + Customer Support")
    print("   Type 'quit' or 'exit' to end the session.")
    print("=" * 60)

    conversation_history = []

    while True:
        try:
            user_input = input("\n👤 You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n\nSession ended.")
            break

        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "bye"):
            print("\n👋 Thank you for shopping with us!")
            break

        # Build multi-turn history
        conversation_history.append({"role": "user", "content": user_input})

        # Agentic loop with full history
        messages = list(conversation_history)
        print("─" * 60)

        while True:
            response = client.messages.create(
                model      = MODEL,
                max_tokens = 2048,
                system     = SYSTEM_PROMPT,
                tools      = TOOLS,
                messages   = messages,
            )
            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "end_turn":
                final_text = "".join(
                    block.text for block in response.content if hasattr(block, "text")
                )
                print(f"\n🤖 Assistant:\n{final_text}")
                # Add assistant turn to persistent history
                conversation_history.append(
                    {"role": "assistant", "content": final_text}
                )
                break

            elif response.stop_reason == "tool_use":
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        print(f"   🔧 [{block.name}] called...")
                        result_str = dispatch_tool(block.name, block.input)
                        tool_results.append({
                            "type"       : "tool_result",
                            "tool_use_id": block.id,
                            "content"    : result_str,
                        })
                messages.append({"role": "user", "content": tool_results})
            else:
                print(f"[Unexpected stop: {response.stop_reason}]")
                break

# Uncomment to run interactive chat:
# chat()
print("✅ chat() function defined — uncomment the last line to run interactively")

✅ chat() function defined — uncomment the last line to run interactively


Cell 8 — Demo: Shopping Scenario 1

"I need a modest evening gown under $300 in size 8. I prefer something on sale."

In [54]:
print("\n" + "═" * 60)
print("🛍️  SHOPPING SCENARIO 1")
print("═" * 60)

response_1 = run_agent(
    "I need a modest evening gown under $300 in size 8. I prefer something on sale."
)


════════════════════════════════════════════════════════════
🛍️  SHOPPING SCENARIO 1
════════════════════════════════════════════════════════════
👤 User: I need a modest evening gown under $300 in size 8. I prefer something on sale.
────────────────────────────────────────────────────────────
🔧 Tool call: search_products({"is_sale": true, "limit": 5, "max_price": 300, "size": "8", "tags": "modest, evening"})
   ↳ Result: {"total_found": 0, "returned": 0, "filters_applied": {"is_sale": true, "limit": 5, "max_price": 300, "size": "8", "tags": "modest, evening"}, "results": []}
🔧 Tool call: search_products({"is_sale": true, "limit": 5, "max_price": 300, "size": "8", "tags": "modest"})
   ↳ Result: {"total_found": 7, "returned": 5, "filters_applied": {"is_sale": true, "limit": 5, "max_price": 300, "size": "8", "tags": "modest"}, "results": [{"product_id": "P0028", "title": "Lumiere Style 28", "vendor": "Lumiere", "price": 226, "compare_at_price": 246, "tags": "sparkle,modest,bridal", "is_

Cell 9 — Demo: Shopping Scenario 2

"I'm attending a prom and need a lace dress in size 10 or 12, budget is $500. Show me the best-selling options."

In [55]:
print("\n" + "═" * 60)
print("🛍️  SHOPPING SCENARIO 2")
print("═" * 60)

response_2 = run_agent(
    "I'm attending a prom and need a lace dress in size 10 or 12, budget is $500. "
    "Show me the best-selling options."
)


════════════════════════════════════════════════════════════
🛍️  SHOPPING SCENARIO 2
════════════════════════════════════════════════════════════
👤 User: I'm attending a prom and need a lace dress in size 10 or 12, budget is $500. Show me the best-selling options.
────────────────────────────────────────────────────────────
⚠️ API Error: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_products{"size": "10", "max_price": 500, "tags": "lace"}</function>\n'}}
🤖 Assistant:
I'm sorry, I encountered a technical error and cannot process your request right now. Details: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_pr

Cell 10 — Demo: Support Scenario 1 (Normal Item, Within Window)

"I want to return order O0005. I bought this dress last week and it doesn't fit."

In [56]:
print("\n" + "═" * 60)
print("📦  SUPPORT SCENARIO 1")
print("═" * 60)

# First let's see what order O0005 looks like
print("[Reference — Order O0005]:")
print(json.dumps(get_order("O0005"), indent=2))
print()

response_3 = run_agent(
    "I want to return order O0005. I bought this dress last week and it doesn't fit. Can I get a refund?"
)


════════════════════════════════════════════════════════════
📦  SUPPORT SCENARIO 1
════════════════════════════════════════════════════════════
[Reference — Order O0005]:
{
  "order_id": "O0005",
  "order_date": "2026-01-31",
  "product_id": "P0001",
  "size": "8",
  "price_paid": 137.0,
  "customer_id": "C024"
}

👤 User: I want to return order O0005. I bought this dress last week and it doesn't fit. Can I get a refund?
────────────────────────────────────────────────────────────
🔧 Tool call: evaluate_return({"order_id": "O0005"})
   ↳ Result: {"order_id": "O0005", "product_id": "P0001", "vendor": "Silk Avenue", "order_date": "2026-01-31", "days_since_order": 94, "is_sale": false, "is_clearance": false, "eligible": false, "return_type": null, "policy_applied": "Normal Item \u2014 14-day window", "reason": "Standard 14-day return window ap...
🤖 Assistant:
I've checked on the status of your order O0005. Unfortunately, the order is 94 days old, which exceeds the 14-day return window for 

Cell 11 — Demo: Support Scenario 2 (Clearance / Vendor Exception)

"I'd like to return order O0001. What are my options?"

In [57]:
print("\n" + "═" * 60)
print("📦  SUPPORT SCENARIO 2")
print("═" * 60)

# Inspect O0001 to understand what we're dealing with
o = get_order("O0001")
p = get_product(o["product_id"])
print(f"[Reference] Order O0001 → Product {o['product_id']}: {p['title']} | "
      f"vendor={p['vendor']} | is_sale={p['is_sale']} | is_clearance={p['is_clearance']}")
print()

response_4 = run_agent(
    "I'd like to return order O0001. The product didn't meet my expectations. What are my options?"
)


════════════════════════════════════════════════════════════
📦  SUPPORT SCENARIO 2
════════════════════════════════════════════════════════════
[Reference] Order O0001 → Product P0015: Lumiere Style 15 | vendor=Lumiere | is_sale=True | is_clearance=False

👤 User: I'd like to return order O0001. The product didn't meet my expectations. What are my options?
────────────────────────────────────────────────────────────
🔧 Tool call: evaluate_return({"order_id": "O0001"})
   ↳ Result: {"order_id": "O0001", "product_id": "P0015", "vendor": "Lumiere", "order_date": "2026-01-28", "days_since_order": 97, "is_sale": true, "is_clearance": false, "eligible": false, "return_type": null, "policy_applied": "Sale Item \u2014 7-day window", "reason": "Sale items can be returned within 7 days...
🔧 Tool call: get_order({"order_id": "O0001"})
   ↳ Result: {"order_id": "O0001", "order_date": "2026-01-28", "product_id": "P0015", "size": "14", "price_paid": 298.0, "customer_id": "C011"}
🤖 Assistant:
I've eva

Cell 12 — Demo: Edge Case — Invalid Order ID

"Can I return order O9999?"* (Does not exist)

In [59]:
print("\n" + "═" * 60)
print("⚠️   EDGE CASE — INVALID ORDER ID")
print("═" * 60)

response_5 = run_agent(
    "I want to return order O9999. Can you process that for me?"
)


════════════════════════════════════════════════════════════
⚠️   EDGE CASE — INVALID ORDER ID
════════════════════════════════════════════════════════════
👤 User: I want to return order O9999. Can you process that for me?
────────────────────────────────────────────────────────────
🔧 Tool call: evaluate_return({"order_id": "O9999"})
   ↳ Result: {"error": "Order 'O9999' not found. Please check the order ID."}
🤖 Assistant:
The order ID 'O9999' was not found. Please check the order ID and try again.


Cell 13 — Demo: Edge Case — Out of Stock Size

In [60]:
print("\n" + "═" * 60)
print("⚠️   EDGE CASE — OUT OF STOCK SIZE")
print("═" * 60)

response_6 = run_agent(
    "I need a cocktail dress in size 20. Budget is $400. Do you have anything?"
)


════════════════════════════════════════════════════════════
⚠️   EDGE CASE — OUT OF STOCK SIZE
════════════════════════════════════════════════════════════
👤 User: I need a cocktail dress in size 20. Budget is $400. Do you have anything?
────────────────────────────────────────────────────────────
⚠️ API Error: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_products{"size": "20", "max_price": "400", "tags": "cocktail"}</function>'}}
🤖 Assistant:
I'm sorry, I encountered a technical error and cannot process your request right now. Details: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_products{"size": "20", "

Cell 14 — Unit Tests for Tool Functions

In [61]:
import traceback

def run_tests():
    passed = 0
    failed = 0

    def check(test_name, condition, details=""):
        nonlocal passed, failed
        if condition:
            print(f"  ✅ PASS: {test_name}")
            passed += 1
        else:
            print(f"  ❌ FAIL: {test_name} — {details}")
            failed += 1

    print("\n" + "═" * 60)
    print("🧪  UNIT TESTS")
    print("═" * 60)

    # ── search_products ──────────────────────────────────────────────────────
    print("\n[search_products]")
    r = search_products({"tags": "modest", "max_price": 300, "size": "8", "is_sale": True})
    check("Returns dict with results key", "results" in r)
    check("All results ≤ $300", all(p["price"] <= 300 for p in r["results"]), r)
    check("All results are on sale", all(p["is_sale"] for p in r["results"]), r)
    check("All results have modest tag",
          all("modest" in p["tags"] for p in r["results"]), r)
    check("All results have stock in size 8",
          all(p["stock_for_requested_size"] not in [0, "0"] for p in r["results"]), r)

    r2 = search_products({"size": "99"})  # Unlikely to have stock
    check("Returns zero results for impossible size 99", r2["total_found"] == 0)

    # ── get_product ──────────────────────────────────────────────────────────
    print("\n[get_product]")
    p = get_product("P0001")
    check("Returns P0001", p.get("product_id") == "P0001")
    check("Has stock_per_size", "stock_per_size" in p and isinstance(p["stock_per_size"], dict))

    p_bad = get_product("INVALID")
    check("Returns error for unknown product", "error" in p_bad)

    # ── get_order ────────────────────────────────────────────────────────────
    print("\n[get_order]")
    o = get_order("O0001")
    check("Returns O0001", o.get("order_id") == "O0001")
    check("Has order_date", "order_date" in o)

    o_bad = get_order("O9999")
    check("Returns error for unknown order", "error" in o_bad)

    # ── evaluate_return ──────────────────────────────────────────────────────
    print("\n[evaluate_return]")

    # Test: invalid order
    ev_bad = evaluate_return("O9999")
    check("Returns error for invalid order", "error" in ev_bad)

    # Test: clearance item → not eligible
    clearance_orders = orders_df[
        orders_df["product_id"].isin(
            inventory_df[inventory_df["is_clearance"] == True]["product_id"]
        )
    ]
    if not clearance_orders.empty:
        cl_order_id = clearance_orders.iloc[0]["order_id"]
        ev_cl = evaluate_return(cl_order_id)
        check(f"Clearance order {cl_order_id} is not eligible",
              ev_cl.get("eligible") == False,
              ev_cl.get("reason", ""))

    # Test: Aurelia Couture → exchange only
    ac_products = inventory_df[inventory_df["vendor"] == "Aurelia Couture"]["product_id"]
    ac_orders = orders_df[orders_df["product_id"].isin(ac_products)]
    if not ac_orders.empty:
        ac_order_id = ac_orders.iloc[0]["order_id"]
        ev_ac = evaluate_return(ac_order_id)
        check(f"Aurelia Couture order {ac_order_id} uses exchange-only policy",
              "Aurelia Couture" in ev_ac.get("policy_applied", ""),
              ev_ac.get("policy_applied", ""))

    print(f"\n{'═'*60}")
    print(f"Results: {passed} passed, {failed} failed")
    print("═" * 60)

run_tests()


════════════════════════════════════════════════════════════
🧪  UNIT TESTS
════════════════════════════════════════════════════════════

[search_products]
  ✅ PASS: Returns dict with results key
  ✅ PASS: All results ≤ $300
  ✅ PASS: All results are on sale
  ✅ PASS: All results have modest tag
  ✅ PASS: All results have stock in size 8
  ✅ PASS: Returns zero results for impossible size 99

[get_product]
  ✅ PASS: Returns P0001
  ✅ PASS: Has stock_per_size
  ✅ PASS: Returns error for unknown product

[get_order]
  ✅ PASS: Returns O0001
  ✅ PASS: Has order_date
  ✅ PASS: Returns error for unknown order

[evaluate_return]
  ✅ PASS: Returns error for invalid order
  ✅ PASS: Clearance order O0003 is not eligible
  ❌ FAIL: Aurelia Couture order O0011 uses exchange-only policy — Clearance — Final Sale

════════════════════════════════════════════════════════════
Results: 14 passed, 1 failed
════════════════════════════════════════════════════════════


In [ ]:
Cell 15 Test Cases

In [62]:
# Quick test — checks everything works end to end
print("=== QUICK DIAGNOSTIC ===")

# Test 1: Tools work
r = search_products(filters={"tags": "modest", "max_price": 300, "size": "8", "is_sale": True})
print(f"✅ search_products works — found {r['total_found']} products")

# Test 2: Order lookup works  
o = get_order("O0001")
print(f"✅ get_order works — {o}")

# Test 3: Return evaluation works
ev = evaluate_return("O0001")
print(f"✅ evaluate_return works — eligible: {ev['eligible']}")

# Test 4: Groq client works
test = client.chat.completions.create(
    model    = MODEL,
    messages = [{"role": "user", "content": "Say hello in one word"}],
    max_tokens = 10,
)
print(f"✅ Groq API works — response: {test.choices[0].message.content}")

print("\n=== ALL SYSTEMS GO — ready to run scenarios ===")

=== QUICK DIAGNOSTIC ===
✅ search_products works — found 7 products
✅ get_order works — {'order_id': 'O0001', 'order_date': '2026-01-28', 'product_id': 'P0015', 'size': '14', 'price_paid': 298.0, 'customer_id': 'C011'}
✅ evaluate_return works — eligible: False
✅ Groq API works — response: Hello

=== ALL SYSTEMS GO — ready to run scenarios ===


In [ ]:
Cell 16 All Possible Scenarios

In [63]:
# ═══════════════════════════════════════════════════════════
# RUN ALL SCENARIOS — Dynamic Agent (No Hardcoding)
# ═══════════════════════════════════════════════════════════

print("\n" + "═" * 60)
print("🛍️  SHOPPING SCENARIO 1")
print("═" * 60)
response_1 = run_agent(
    "I need a modest evening gown under $300 in size 8. I prefer something on sale."
)

print("\n" + "═" * 60)
print("🛍️  SHOPPING SCENARIO 2")
print("═" * 60)
response_2 = run_agent(
    "I am attending a prom and need a lace dress in size 10, budget is $500. Show me best-selling options."
)

print("\n" + "═" * 60)
print("📦  SUPPORT SCENARIO 1 — Normal Return")
print("═" * 60)
response_3 = run_agent(
    "I want to return order O0005. I bought this dress last week and it doesn't fit. Can I get a refund?"
)

print("\n" + "═" * 60)
print("📦  SUPPORT SCENARIO 2 — Vendor Exception")
print("═" * 60)
response_4 = run_agent(
    "I would like to return order O0001. The product did not meet my expectations. What are my options?"
)

print("\n" + "═" * 60)
print("⚠️   EDGE CASE 1 — Invalid Order ID")
print("═" * 60)
response_5 = run_agent(
    "Can you process a return for order O9999?"
)

print("\n" + "═" * 60)
print("⚠️   EDGE CASE 2 — Out of Stock Size")
print("═" * 60)
response_6 = run_agent(
    "I need a cocktail dress in size 20. Budget is $400. Do you have anything?"
)

print("\n" + "═" * 60)
print("✅  ALL SCENARIOS COMPLETE")
print("═" * 60)


════════════════════════════════════════════════════════════
🛍️  SHOPPING SCENARIO 1
════════════════════════════════════════════════════════════
👤 User: I need a modest evening gown under $300 in size 8. I prefer something on sale.
────────────────────────────────────────────────────────────
⚠️ API Error: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_products{"is_sale": true, "size": "8", "max_price": 300, "tags": "modest, evening"}</function>'}}
🤖 Assistant:
I'm sorry, I encountered a technical error and cannot process your request right now. Details: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_products{